# Step 4 — ESIM on `overture_case2s`

**Why:** the word2vec baseline (`01_word2vec.ipynb`) averages each address into one vector, so it cannot tell `MAIN ST` from `MAPLE ST` (case2s AUC ≈ 0.6). ESIM lets the two addresses look at each other's words before the comparison.

**Spec (2026-11-02):** frozen word2vec (200-d, skip-gram, 50 epochs) with learned `UNK` and masked `PAD` → BiLSTM (200) → soft alignment both ways (PAD masked with −∞) → `[a; ã; a−ã; a⊙ã]` → composition BiLSTM (200) → masked mean + max pooling → `[v_a; v_b]` → MLP (200, dropout 0.3) → match probability. Training protocol identical to Step 3 (BCE, Adam 1e-3, `is_better` with the case1 guardrail, patience 3, max 20 epochs). Symmetry: original `[v_a; v_b]`, swap test later.

**Build plan:** all layers first, one shape pass through the whole model, then three tests (PAD mask, softmax axis, tiny-set overfit).

**Layout (2026-11-14):** every component lives once, in `model.py` (tokenize → tables → batches → layers → `ESIM`); training and the readers of saved runs in `training.py`. Each section below names the code it uses, then tests it. Merged here from `03_esim_extension.ipynb`: the two extension settings (§12b), all-five training (§13) and results (§14e–g), and three diagnostics (D6–D8).

## 0. Setup
**Why:** one place for imports and the constants every cell uses.

In [3]:
# Setup: imports and constants
%load_ext autoreload
%autoreload 2

import os
import sys
import time
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from gensim.models import FastText, Word2Vec

from helpers.pairs import load_pairs
from helpers.check_model_performance import review_typical_slices, find_random_sample_given_mask
from siamese_model.model import PairDataset   # word-agnostic: returns raw strings
from siamese_model.training import train_epoch, predict   # same batch format as our collate
from siamese_model.training import evaluate, broken_run_problems, THRESHOLD, CASE1_GUARDRAIL
from esim.model import (tokenize, PAD_ID, UNK_ID, EMB_DIM, HIDDEN, DROPOUT, BATCH_SIZE, MAX_LEN,
                        plain_table, random_rows_table, stacked_table, encode_batch, make_collate, make_loaders,
                        EmbeddingLayer, ContextEncoder, attention_weights, align, enhance,
                        Composition, MeanMaxPooling, ClassifierHead, ESIM)
from esim.training import (LR, MAX_EPOCHS, PATIENCE, SEEDS, SETTINGS, run_training, save_run,
                           health, slice_table, rare_number_group, rare_number_table)

DATASET = "overture_case2s"

# Embedding setting for the build sections (§2–§12) and 14a/14b: which table the step-by-step tests use
#   W2V_KIND: "word2vec" or "fasttext" (fastText = word2vec + char 3-6-gram vectors)
#   W2V_MIN_COUNT: 2 = original run (rare house numbers -> UNK), 1 = every word gets a vector
#   all five settings and their run folders: SETTINGS in training.py; §13 trains whatever is missing
W2V_KIND = "word2vec"
W2V_MIN_COUNT = 2
W2V_PATH = f"checkpoints/{W2V_KIND}_50_mc{W2V_MIN_COUNT}.model"      # from 01_word2vec.ipynb
CHECKPOINT_DIR = "checkpoints/esim_" + ("ft_" if W2V_KIND == "fasttext" else "") + f"mc{W2V_MIN_COUNT}"   # runs + saved predictions (git-ignored)

pd.set_option("display.width", 250)
pd.set_option("display.expand_frame_repr", False)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Load the pairs
**Why:** same pair files and shared schema as word2vec and Step 3.
**Does:** loads train / val / test. Test stays untouched until the final evaluation.
**Output:** `train`, `val`, `test` DataFrames and their label arrays

In [4]:
# Load: train / val / test pairs
train, train_label = load_pairs(DATASET, "train")
val, val_label = load_pairs(DATASET, "val")
test, test_label = load_pairs(DATASET, "test")
splits = {"train": (train, train_label), "val": (val, val_label), "test": (test, test_label)}
all_addr = pd.concat([d[c] for d in (train, val, test) for c in ("address_a", "address_b")])
print(len(train), len(val), len(test))

202568 41865 43844


## 2. The word2vec model
**Why:** ESIM's word vectors come from the 50-epoch word2vec model. `01_word2vec.ipynb` never saved its model, and a retrained one differs run to run (Python's string hashing is randomised per kernel start), so it should be trained once and saved.
**Does:** gets the trained model into this notebook (gensim can save a model to a file and load it back; where the training and saving happen is your call).
**Output:** `w2v`, a trained gensim model with 200-d vectors

In [5]:
# word2vec model: train once and save / load
EMB_DIM = 200
w2v = (FastText if W2V_KIND == "fasttext" else Word2Vec).load(W2V_PATH)   # set in Setup
assert w2v.wv.vector_size == EMB_DIM
print(w2v.epochs, len(w2v.wv.key_to_index))

50 4282


## 3. Vocabulary and embedding matrix
- **Background**: Even though our `word2vec` model technically does the id-encoding as well as vector-encoding, we will split it up into two stage. `ID-Conversion` and `Vector-Conversion`. `Vector-Conversion` happens in this layer. `ID-Conversion` occurs in vocab
- Before we input anything, our `DataLoader` will upload all word_sequences as a series of **id's**, not words
- Then: in our embedding layer, we will take the embedding table from our trained `word2vec` model and place it into `nn.Embedding`. 
- **A Note on nn.Embeddings**: We often think of embeddings as being similar to a dict, but it's more like a list. If we give it a list of embeddings, the key will be the `index` and the value will be the embedding itself. So when we initially add two `zero-embeddings`, it essentially assign `0,1` to a zero embedding. 
- **What are we training on?**: We want our `word2vec` embeddings to be frozen, but we want to have a trainable `<unk>` parameter, because we any arbitrary embedding, will likely implicitly mean something on the embedding space if not trained. We get both by creating an `Embedding Layer` whose **weights are frozen**, which avoids updating gradients, while assigning `<unk>` using `Parameter` to allow only `<unk>` embeddings to be updated

##### Forward Dorward
- When we initially run our id through the embedding layer we get `out` of size $(n, m, h)$
- Our `is_unk` creates a mask of size $(n, m, 1)$, and essentially serves as a boolean mask
- `self.unk` is of size $(h)$, but when used will be broadcasted to size $(n,m,h)$
- when we use `torch.where`, 
    - `is_unk`, serves as the boolean, of which value should we use. 
    - `self.unk` is used as a constant value if `1`
    - `out` serves as a `mapping_function`
    - The reason `self.unk` and `out` work differently is because of their sizes (partially due to broadcasting), affect their output

**Code:** `model.py` → `plain_table` (vocab + matrix), `EmbeddingLayer`.

In [6]:
# Vocab with PAD / UNK, embedding matrix (the +2 shift: rows 0 / 1 are PAD / UNK)
vocab, matrix = plain_table(W2V_KIND, W2V_MIN_COUNT)
assert len(vocab) == len(w2v.wv.key_to_index) + 2

Embedding checks (written by Claude, 2026-11-04): each word gets its own word2vec row (the +2 shift), `PAD` gives zeros, `UNK` gives the learned `unk`. `unk` is set to ones first, because at init it equals the zero `PAD` row and the `UNK` check couldn't tell them apart.


In [7]:
# Embedding checks: word -> its own word2vec row, PAD -> zeros, UNK -> unk
emb = EmbeddingLayer(matrix)
with torch.no_grad():
    emb.unk.fill_(1.0)

n_words = len(w2v.wv.index_to_key)
words = [w2v.wv.index_to_key[i] for i in (0, 1, n_words // 2, n_words - 1)]   # first, second, middle, last
for w in words:
    assert torch.equal(emb(torch.tensor([[vocab[w]]]))[0, 0], torch.tensor(w2v.wv[w])), w
assert (emb(torch.tensor([[PAD_ID]])) == 0).all()
assert torch.equal(emb(torch.tensor([[UNK_ID]]))[0, 0], emb.unk)
print("embedding checks passed on", words)


embedding checks passed on ['ST', 'RD', '1096', '1586']


## 4. Encode a batch
- We just tokenize our variables
- We have to minimize/limit size to avoid batch-size blowup (This more of concern when dealing with `chars`, but not so much for words)
- We pad it so they can be turned into a 2D tensor

**Code:** `model.py` → `encode_batch` (`MAX_LEN = 85`).

- We were considering that an empty string could silently break gradients, test shows we have no empty addresses, so this is a non-issue

In [8]:
all_addr = pd.concat([d[c] for d in (train, val, test) for c in ("address_a", "address_b")])
empty = all_addr[all_addr.map(tokenize).str.len() == 0]
print(len(empty)); empty.value_counts().head()


0


Series([], Name: count, dtype: int64)

## 5. Dataset and DataLoaders
**Why:** feeds the model one batch at a time, encoding each batch only when it is drawn.
**Does:** a `Dataset` returning raw address strings and the label, a collate function that encodes both sides, and loaders (train shuffled, val and test not).
**Output:** `train_loader`, `val_loader`, `test_loader`

**Code:** `model.py` → `make_collate`, `make_loaders` (`PairDataset` from `siamese_model/model.py`).

In [9]:
loaders = make_loaders(vocab, splits)   # train shuffled, val / test not
train_loader, val_loader, test_loader = loaders["train"], loaders["val"], loaders["test"]
train_dataset = train_loader.dataset    # used by the timing cell

## 6. Encode: BiLSTM
- **BiLSTM**: its output_size is called `hidden_size` NOT `OUTPUT_SIZE` because the resultant sequence is just them dragging along the short-term memory at each timestep
- **dropout**: dropout is a little interesting and does not need to specify size, because it's not a layer w/ weights. It's merely a `mask`, whose size is equivelant to the **input_dims**
    - **But can't we infer input size for all things?**: We can, which is why we also have `nn.LazyLinear` which does exactly that. The ouput_size, however, cannot be inferred, and is a design choice.

**Code:** `model.py` → `ContextEncoder` (dropout → shared `BiLSTM` from `siamese_model/model.py`).

## 7. Align: cross attention
- **What does @ do?**: `@` understands that for 3d operations with size (n, m_a, h) @ (n, h, m_b) that `n` is representative of number of rows, which means instead of 3D dot-product, we instead n operations of (m_a, b) x (h, m_b) 
    - This is also why we do `.transpose(1,2)`, which swaps `dim=1` and `dim=2`
    - The result shape is (B, m_a, m_b)
- **How do we create the Intial Mask?**: 
    - When we do B.size(1), we create w/ `torch.arange(B.size(1))`, our output_size is (m_b)
    - Next `len_b` is initially (n), but after `[:, None]` it becomes (n, 1)
    - Now we have to deal with `broadcasting`. After broadcasting:
        - `arange`: (m_b) -> (1, m_b) -> (n, m_b)
        - `len_b`: (n, 1) -> (n, m_b)
        - `result`: (n, m_b)
        - **NOTE**, we initially add 1 to right of `len_b`, cause broadcasting only attempts to add 1s to left
    - We can think of it like a `dict` where each tensor of length `seq_b_length` corresponds to a row in `rows`
- **How do we do Filled Mask?**
    - Firstly we handle mask whose shape is current (n, m_b) becomes (n, 1, m_b)
        - We use `~` to get the inverse because `original_mask` includes all values who are *within the length*
    - Now, we need to update mask to match the size of e. In order to understand, we can think of the initial mask representing a `dict`. That dict says for THIS ROW, we only use these versions of `seq_b`. Then, in order to apply it to the `attention_matrix`, we just need to think, now just apply it and tell all `seq_a` to ignore these values for `seq_b`
    - `softmax`takes the last dimension (which is `dim=2`), and makes it so their sum becomes 1

**Code:** `model.py` → `attention_weights`, `align`.

A check to ensure that A_tilde produced by `align` gives the same shape as `original_A`

In [10]:
emb, enc = EmbeddingLayer(matrix), ContextEncoder(matrix.shape[1])
ids_a, len_a, ids_b, len_b, y = next(iter(train_loader))

A = enc(emb(ids_a), len_a)
B = enc(emb(ids_b), len_b)

A_tilde = align(A, B, len_b)
B_tilde = align(B, A, len_a)
assert A_tilde.shape == A.shape and B_tilde.shape == B.shape

We want to ensure that our attention `softmax` actually runs on the correct_dimension (dim = 2).
- This is important cause otherwise `softmax` can succeed while silently running on the wrong axis, messing up our `attention`/model

In [11]:
mask_index = 4
X, Y = torch.randn((1,8, 5)), torch.randn((1,8, 5))   # not A / B: those names hold the real batch used by later checks
len_Y = torch.tensor([mask_index])
res = attention_weights(X, Y, len_Y)
assert torch.allclose(res[:, :, :mask_index].sum(-1), torch.ones(res.shape[:2]), atol = 1e-6)
assert torch.allclose(res[:, :, mask_index:].sum(-1), torch.zeros(res.shape[:2]), atol = 1e-6)

## 8. Enhance
**Why:** puts match and mismatch signals side by side so later layers can read them.
**Does:** per word, concatenates `a`, `ã`, `a − ã` and `a ⊙ ã`.
**Output:** `M_a`, `M_b` (batch × length × 4H)

**Code:** `model.py` → `enhance`.

1) Check that shape is same size except for embedding which should be 4 times larger
2) first 400 (2 * EMB_DIM) should be `original_A`
3) These tensors have `require_grad`, meaning they can be linked back to the model

In [12]:
M_a, M_b = enhance(A, len_a, B, len_b)
assert M_a.shape == (*A.shape[:2], 4 * A.size(-1))       # (n, m_a, 1600)
assert torch.equal(M_a[..., :400], A)                    # first block is A itself
assert M_a.requires_grad                                 # gradient history kept


## 9. Compose: second BiLSTM
**Why do we only process one side?**: We only process one side because later inside the `ESIM Module`, we will run `enhance` and run `Composition` per word_sequence then
- Add `relu` to avoid running a simple linear function

**Code:** `model.py` → `Composition`.

- Check that the output embedding size matches twice the embedding dimension
- Make sure that BiLSTM padded comparisons are not computed and are `0`

In [13]:
comp = Composition()
C_a = comp(M_a, len_a)
C_b = comp(M_b, len_b)
assert C_a.shape == (*M_a.shape[:2], HIDDEN * 2)
assert (C_a[0, len_a[0]:] == 0).all()

## 10. Pool
**Output Notes**: 
- (B, 800), since word_sequences get compressed
- concatenated `mean/max` into same embedding

**Code:** `model.py` → `MeanMaxPooling`.

In [14]:
pooling = MeanMaxPooling()
v_a, v_b = pooling(C_a, len_a), pooling(C_b, len_b)
assert v_a.shape == (C_a.size(0), 4 * HIDDEN)                     # mean + max of 2 * HIDDEN each
assert torch.isfinite(v_a).all() and torch.isfinite(v_b).all()   # a -inf from the max mask would show here

## 11. Head and the assembled model
- Pretty Standard just squash both layers on top of each other before sending to `ClassifierHead`

**Code:** `model.py` → `ClassifierHead`, `ESIM`.

Check that the classifier_head, generates a 1D tensor whose length equals number of rows

In [15]:
head = ClassifierHead()
x = torch.cat([v_a, v_b], dim = -1)
logits = head(x)
assert logits.shape == (x.size(0),)

In [16]:
esim = ESIM(matrix)
ids_a, len_a, ids_b, len_b, y = next(iter(train_loader))
res = esim(ids_a, len_a, ids_b, len_b)
assert res.shape == (ids_a.size(0),)

## 12. One pass and tests
**Why:** shape bugs crash, but a missing PAD mask or a wrong softmax axis only makes the numbers worse, so those need explicit checks.
**Does:** (a) one batch through the whole model: the shape assert at the end of §11; (b) mask test: the same pair padded to two different lengths gives the same output; (c) axis check: each row of the attention weights sums to 1 — moved to §7; (d) overfit test: a tiny set reaches near-zero loss.
**Output:** passing asserts

#### B) Mask Test
Testing if our end-to-end code can properly filter out padding tokens
- make sure to set to `eval` to not use `dropout`. By default starts in `train` mode
- use `torch.allclose` to compare since dot-products will not return exact result

In [17]:
# Shape pass and tests

def modified_ids(ids, lengths):
    mask = torch.arange(ids.size(1)) >= lengths[:, None]   # position == length is the first PAD
    new_ids = torch.where(mask, 2, ids)
    return new_ids

# Testing Masking End-to-End
orig_ids = [[2,0,0], [2,2,0], [1,2,3]]
orig_lens = [1,2,3]

esim = ESIM(matrix).eval()
ids_a, len_a = torch.tensor(orig_ids), torch.tensor(orig_lens)
ids_b, len_b = torch.tensor([[3,4,5], [4,0,0], [5,3,0]]), torch.tensor([3,1,2])   # different lengths from A, so a len_a/len_b swap would fail
mod_a, mod_b = modified_ids(ids_a, len_a), modified_ids(ids_b, len_b)

assert torch.allclose(esim(ids_a, len_a, ids_b, len_b), esim(mod_a, len_a, mod_b, len_b), atol = 1e-6)


#### D) Overfit Test
- Unlike in normal training, our goal is to overfit the set
- This is so we're able to test the gradients of our model and assert the MODEL CAN LEARN

In [18]:
toy = pd.DataFrame([
    ("12 main st",        "12 main st",         1),   # identical
    ("12 main st",        "12 main street",     1),   # abbreviation
    ("45 oak avenue",     "45 oak ave",         1),   # abbreviation
    ("301 elm road",      "301 elm raod",       1),   # typo
    ("88 pine st apt 2",  "88 pine st",         1),   # extra unit (building-level match)
    ("12 main st",        "14 main st",         0),   # case1: same street, different number
    ("45 oak avenue",     "47 oak avenue",      0),   # case1
    ("301 elm road",      "301 elm street",     0),   # case2-style: same number, different street type
    ("12 main st",        "990 harbor view dr", 0),   # easy
    ("88 pine st",        "5 quarry lane",      0),   # easy
], columns=["address_a", "address_b", "label"])
toy_labels = toy["label"]

toy_ds = PairDataset(toy, toy_labels)
toy_loader = DataLoader(toy_ds, batch_size=len(toy_ds), shuffle=False, collate_fn=make_collate(vocab))   # one batch = the whole toy set

#TEST: Checks to make sure our dataset is not accidentally filled with words that work is <unk> on encoder
[w for s in pd.concat([toy.address_a, toy.address_b]) for w in tokenize(s) if w not in vocab]

['APT', 'RAOD']

In [19]:
model = ESIM(matrix)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()
for epoch in range(300):
    loss = train_epoch(model, toy_loader, opt, loss_fn)
    if epoch % 50 == 0:
        print(f"epoch {epoch:3d} loss {loss:.4f}")
assert loss < 0.05

epoch   0 loss 0.6929
epoch  50 loss 0.0001
epoch 100 loss 0.0000
epoch 150 loss 0.0001
epoch 200 loss 0.0000
epoch 250 loss 0.0000


#### E) Frozen vs learned embedding 
- One training step must leave the word2vec rows unchanged and move `unk`
- Needs `UNK` words in the batch, or `unk` gets no gradient: the toy set has `APT` and `RAOD`


In [20]:
# One training step: word2vec rows unchanged, unk moved
model = ESIM(matrix)
frozen_before = model.embedding.frozen.weight.clone()
unk_before = model.embedding.unk.detach().clone()
opt = torch.optim.Adam(model.parameters(), lr=LR)
train_epoch(model, toy_loader, opt, nn.BCEWithLogitsLoss())
assert torch.equal(model.embedding.frozen.weight, frozen_before), "word2vec rows changed: not frozen"
assert not torch.equal(model.embedding.unk.detach(), unk_before), "unk did not move: no gradient reached it"


#### F) Reproduce check: `model.py`'s ESIM reproduces the saved predictions
(From `03_esim_extension.ipynb` §1, where it checked the module against 02's own copy of the model. Now there is one copy; the check still guards against an edit to `model.py` silently changing a saved run.)
- Claude made some changes to model, ensuring no changes affected the preds
- Test current model weights with previous output

In [21]:
vocab_mc2, matrix_mc2 = plain_table("word2vec", 2)
ckpt = torch.load("checkpoints/esim_mc2/esim_seed_0.pt")
assert ckpt["vocab"] == vocab_mc2, "vocab differs from the one the run trained with"

model = ESIM(matrix_mc2)
model.load_state_dict(ckpt["state_dict"])
val_loader_mc2 = make_loaders(vocab_mc2, {"val": splits["val"]})["val"]
preds = (predict(model, val_loader_mc2) >= THRESHOLD).int().numpy()
saved = np.load("checkpoints/esim_mc2/preds_val_seed_0.npy")
assert np.array_equal(preds, saved), f"{(preds != saved).sum()} predictions differ from the saved ones"
print("module ESIM reproduces 02 (mc2, seed 0, val):", len(preds), "predictions identical")

module ESIM reproduces 02 (mc2, seed 0, val): 41865 predictions identical


## 12b. Tables for the two extension settings (`rand_mc2`, `stack_mc1`)
**Why:** with word2vec `min_count=2`, every rare house number becomes the same `UNK`, so a same-street case1 pair with two rare numbers is the same input as an identical pair with a rare number (02, D1–D4). fastText fixed that but cost ~10 pts on case2s (02, D5). This notebook tests two settings that target the numbers without (option 5) or while also (option 4) changing how street words are represented.

**Code:** written by Claude (2026-11-06, at David's request, in `03_esim_extension.ipynb`; merged here 2026-11-14; David does the results analysis). Tables: `model.py` → `random_rows_table`, `stacked_table`; the model is the same `ESIM` with the table as an argument.

| | option 5: random rows | option 4: stacked |
|---|---|---|
| setting name | `rand_mc2` | `stack_mc1` |
| source vectors | word2vec `mc2` | word2vec `mc1` and fastText `mc1` |
| vocab | `PAD` 0, `UNK` 1, word2vec words (+2), then every token word2vec dropped | `PAD` 0, `UNK` 1, every word2vec `mc1` word (+2) |
| matrix rows | 2 zero rows + word2vec rows + one random row per dropped token | 2 zero rows + `[w2v.wv[w]; ft.wv[w]]` per word |
| random rows | normal, per-dimension mean / std of the word2vec rows, seed 0, frozen, same for all training seeds | — |
| embedding size | 200 | 400 (context BiLSTM input 400, hidden stays 200) |
| run folder | `checkpoints/esim_rand_mc2/` | `checkpoints/esim_stack_mc1/` |
| protocol | as 02: 3 seeds, `is_better` (case1 guardrail 98), patience 3, val/test predictions saved | same |

**Expectation:** option 5 gives `3015` its own fixed vector (same on both sides, unrelated to `3017`) without changing street words → numbers fixed, case2s near `mc2`. Option 4 hands the model both signals; the fastText half brings look-alike streets close again, so case2s depends on whether training learns to ignore it.

### Option 5: word2vec `mc2` + one fixed random row per dropped token
- `random_rows_table` creates the data needed for running random row. 
- We do so by figuring out which values are included in word2vec(mc=2) vs in vocab, and then creating a corresponding vocab and matrix.
    - vocab means what `word -> id`
    - matrix mean what we use for the embedding table
- **NOTE** We create our random values by using the std dev of our wv.vectors, which means that our random values look similar to preexisting embeddings

In [22]:
vocab_rand, matrix_rand, dropped = random_rows_table(all_addr, min_count=2, seed=0)
n_w2v = len(vocab_mc2) - 2
print(f"word2vec rows: {n_w2v}, random rows: {len(dropped)}, matrix: {tuple(matrix_rand.shape)}")
print("dropped, all-digit:", sum(t.isdigit() for t in dropped), "| sample:", dropped[:5], dropped[-5:])

word2vec rows: 4282, random rows: 1868, matrix: (6152, 200)
dropped, all-digit: 699 | sample: ['004', '01', '02', '03', '06033'] ['YAWBUX', 'YOUNG', 'YOUNGS', 'ZLOTNICK', 'ZUMBACH']


### Double Check Random Row Formulation
1) no `UNK` left in any value for all_addr
2) rare number example doesn't occur i.e. when running on the same scenario, it doesn't show something like `UNK MAIN ST | UNK MAIN ST`
3) Assert that for two of random dropped values, which didn't pass the threshold, are both different and sufficiently disimilar 
4) Since random matrix is just normal matrix, with additional random matrices added for `UNK` indicies, verify that's actually the case, by checking equal for `matrix_rand[2 + n_w2v:]`

In [23]:
# no UNK left in any split
assert all(t in vocab_rand for s in all_addr for t in tokenize(s)), "some token still maps to UNK"

# same string -> same row, on both sides of an identical pair with a dropped number
grp = rare_number_group(val)
row = val[grp & (val["address_a"] == val["address_b"]).values].iloc[0]
ids_a, _ = encode_batch([row["address_a"]], vocab_rand)
ids_b, _ = encode_batch([row["address_b"]], vocab_rand)
assert torch.equal(ids_a, ids_b) and ids_a[0, 0] != UNK_ID and ids_a[0, 0] >= 2 + n_w2v, "rare number not on its own random row"
print("identical pair:", row["address_a"], "->", ids_a.tolist())

# different dropped tokens -> different rows
nums = [t for t in dropped if t.isdigit()][:2]
r0, r1 = matrix_rand[vocab_rand[nums[0]]], matrix_rand[vocab_rand[nums[1]]]
assert not torch.allclose(r0, r1)
print(f"{nums[0]} vs {nums[1]}: cos {torch.cosine_similarity(r0, r1, dim=0):.2f}")

# word2vec rows = 02's table; random rows on the same scale
assert torch.equal(matrix_rand[: 2 + n_w2v], matrix_mc2)
print("row norm, word2vec mean:", matrix_mc2[2:].norm(dim=1).mean().item(), "| random mean:", matrix_rand[2 + n_w2v:].norm(dim=1).mean().item())

identical pair: 4536 CONGRESS ST -> [[4924, 866, 2]]
004 vs 01: cos 0.27
row norm, word2vec mean: 2.8631844520568848 | random mean: 3.1257388591766357


### Option 4: stacked `[word2vec mc1; fastText mc1]`
- We basically run `hstack` so that our embeddings now get `[w2v_vector, ft_vectors]`, and that's all we need

In [24]:
vocab_stack, matrix_stack = stacked_table(min_count=1)
print(f"rows: {len(vocab_stack)}, matrix: {tuple(matrix_stack.shape)}")

rows: 6151, matrix: (6152, 400)


- We will be running on `mc1`, so we're just checking to make sure `mc1` has no corresponding gaps
- Also verifying that the stacked embedding is the same as the `word2vec` embedding stacked on top of the `fast-text` embedding

In [25]:
from gensim.models import FastText, Word2Vec
w2v1 = Word2Vec.load("checkpoints/word2vec_50_mc1.model").wv
ft1 = FastText.load("checkpoints/fasttext_50_mc1.model").wv

assert all(t in vocab_stack for s in all_addr for t in tokenize(s)), "some token still maps to UNK"
for w in ["MAIN", "3015", "ST"]:
    expected = torch.cat([torch.tensor(w2v1[w]), torch.tensor(ft1[w])])
    assert torch.allclose(matrix_stack[vocab_stack[w]], expected), w
print("stacked rows = [word2vec; fastText] for MAIN, 3015, ST")

stacked rows = [word2vec; fastText] for MAIN, 3015, ST


A verbatim test to just ensure that the initial embeddings are FROZEN and that the UNK embedding is NOT FROZEN

In [26]:
small_train = PairDataset(train.iloc[:4 * BATCH_SIZE], train_label.iloc[:4 * BATCH_SIZE])   # 4 batches is enough
for name, vocab_s, matrix_s in [("rand_mc2", vocab_rand, matrix_rand), ("stack_mc1", vocab_stack, matrix_stack)]:   # _s: keep §3's vocab / matrix
    torch.manual_seed(0)
    model = ESIM(matrix_s)
    table_before = model.embedding.frozen.weight.clone()
    head_before = model.head.linear2.weight.detach().clone()
    loader = DataLoader(small_train, batch_size=BATCH_SIZE, collate_fn=make_collate(vocab_s))
    train_epoch(model, loader, torch.optim.Adam(model.parameters(), lr=1e-3), nn.BCEWithLogitsLoss())
    assert torch.equal(model.embedding.frozen.weight, table_before), f"{name}: table changed"
    assert not torch.equal(model.head.linear2.weight, head_before), f"{name}: nothing trained"
    print(name, "table frozen, head trained")

rand_mc2 table frozen, head trained
stack_mc1 table frozen, head trained


## 13. Training
- Timing and Building training loop

In [27]:
SEED = 0
N_TIMING_BATCHES = 200

torch.manual_seed(SEED)
timing_model = ESIM(matrix)
timing_opt = torch.optim.Adam(timing_model.parameters(), lr=LR)
subset_loader = DataLoader(Subset(train_dataset, range(N_TIMING_BATCHES * BATCH_SIZE)), batch_size=BATCH_SIZE, collate_fn = make_collate(vocab))

# train/val loaders come from §2 (train shuffled); don't rebuild them here

t0 = time.perf_counter()
train_epoch(timing_model, subset_loader, timing_opt, nn.BCEWithLogitsLoss())
train_seconds = time.perf_counter() - t0
print(f"train: {train_seconds / N_TIMING_BATCHES:.3f} s/batch -> {train_seconds / N_TIMING_BATCHES * len(train_loader) / 60:.1f} min/epoch")

t0 = time.perf_counter()
predict(timing_model, val_loader)
print(f"val pass: {time.perf_counter() - t0:.1f} s")

train: 0.060 s/batch -> 3.2 min/epoch
val pass: 11.6 s


**Code:** `training.py` → `run_training` (seeded run, early stopping on `is_better`, patience 3), `save_run` (checkpoint, history, val/test predictions).

All five settings, one loop. Existing runs are skipped, so re-running only trains what is missing. About 3 min per epoch per seed for `rand_mc2`; `stack_mc1`'s first BiLSTM is wider.
- `mc2`, `mc1`, `ft_mc1` were trained 2026-11-05/06 by 02's earlier per-setting loop (the `W2V_KIND` / `W2V_MIN_COUNT` switch in Setup), same protocol.
- `rand_mc2`, `stack_mc1` were trained 2026-11-06 in `03_esim_extension.ipynb`; the saved output below is from those runs.

In [28]:
tables = {   # setting -> builder of (vocab, matrix, config); built only if a seed is missing
    "mc2":       lambda: (*plain_table("word2vec", 2), {"w2v_kind": "word2vec", "w2v_min_count": 2}),
    "mc1":       lambda: (*plain_table("word2vec", 1), {"w2v_kind": "word2vec", "w2v_min_count": 1}),
    "ft_mc1":    lambda: (*plain_table("fasttext", 1), {"w2v_kind": "fasttext", "w2v_min_count": 1}),
    "rand_mc2":  lambda: (vocab_rand, matrix_rand, {"table": "word2vec_50_mc2 + random rows (seed 0)"}),
    "stack_mc1": lambda: (vocab_stack, matrix_stack, {"table": "[word2vec_50_mc1; fasttext_50_mc1]"}),
}
for name, build in tables.items():
    folder = SETTINGS[name]
    missing = [seed for seed in SEEDS if not os.path.exists(os.path.join(folder, f"esim_seed_{seed}.pt"))]
    if not missing:
        continue
    vocab_s, matrix_s, config = build()
    loaders_s = make_loaders(vocab_s, splits)
    for seed in missing:
        print(f"{'='*20} {name} seed {seed} {'='*20}")
        history, best, best_state = run_training(matrix_s, loaders_s, val, val_label, seed)
        save_run(folder, seed, history, best, best_state, vocab_s, matrix_s, loaders_s,
                 {**config, "emb_dim": matrix_s.shape[1], "dataset": DATASET})
        print(f"best: {best}\nproblems: {broken_run_problems(history)}")


## 14. Results per slice
**Why:** the headline question is whether alignment fixes case2s.
**Does:** val per slice against the word2vec baseline and the Step 3 siamese model; test once, at the end, for the best configuration; row added to `results.md`.
**Output:** slice table

### Run health, all five settings
- Check for any health problems, which is the same as `broken_run_problems`

In [29]:
# health: epochs trained and broken_run_problems per run ([] = healthy)
health()

epochs problems
setting   seed                 
mc2       0         11       []
          1          9       []
          2          8       []
mc1       0          5       []
          1         12       []
          2          8       []
ft_mc1    0          5       []
          1          8       []
          2          8       []
rand_mc2  0          5       []
          1         20       []
          2          8       []
stack_mc1 0         10       []
          1          9       []
          2          7       []

### 14a. Val, per seed 
- Only real concern was the low match rate for positive:identicals, where they were unusually low
    - Also, seed 2 surprising got `100%` right somehow
- This leads us to figuring out that Word2Vec is mishandling similar numbers, where it isn't able to effectively tell the difference between addresses where only the house_number is potentially different

In [30]:
# 14a: val per seed
res_val = {}
for seed in [0, 1, 2]:
    print(f"{'='*20} seed:{seed} {'='*20}")
    ckpt = torch.load(os.path.join(CHECKPOINT_DIR, f"esim_seed_{seed}.pt"))

    model = ESIM(matrix)
    model.load_state_dict(ckpt["state_dict"])
    # res = evaluate(model, test_loader, ot_test, ot_test_label)
    # print(res)
    with torch.no_grad():
        val_preds = (predict(model, val_loader) >= THRESHOLD).int().numpy()
        res_val[seed] = val_preds
        np.save(os.path.join(CHECKPOINT_DIR, f"preds_val_seed_{seed}.npy"), val_preds)   # read by 14d
        val_slices = review_typical_slices(val, val_preds, val_label)
        print(val_slices)

==================== seed:0 ====================
                                      slice      n  correct  errors % correct
0                      positives: identical   3390     3341      49     98.6%
1                  positives: non-identical    797      659     138     82.7%
2     neg: case1 (same street, diff number)  13781    13545     236     98.3%
3  neg: case2 (same number, similar street)     34       19      15     55.9%
4                         neg: case3 (easy)  22969    22951      18     99.9%
5   neg: case2s (swapped look-alike street)    894      680     214     76.1%
6                        overall (accuracy)  41865    41195     670     98.4%
==================== seed:1 ====================
                                      slice      n  correct  errors % correct
0                      positives: identical   3390     3320      70     97.9%
1                  positives: non-identical    797      652     145     81.8%
2     neg: case1 (same street, diff number) 

KeyboardInterrupt: 

### 14b. Test, all 3 seeds
- Same problem as val, but problem is exacerbated.

In [ ]:
res_test = {}
for seed in [0, 1, 2]:
    print(f"{'='*20} seed:{seed} {'='*20}")
    ckpt = torch.load(os.path.join(CHECKPOINT_DIR, f"esim_seed_{seed}.pt"))

    model = ESIM(matrix)
    model.load_state_dict(ckpt["state_dict"])
    # res = evaluate(model, test_loader, ot_test, ot_test_label)
    # print(res)
    with torch.no_grad():
        test_preds = (predict(model, test_loader) >= THRESHOLD).int().numpy()
        res_test[seed] = test_preds
        np.save(os.path.join(CHECKPOINT_DIR, f"preds_test_seed_{seed}.npy"), test_preds)   # read by 14d
        test_slices = review_typical_slices(test, test_preds, test_label)
        print(test_slices)

==================== seed:0 ====================


                                      slice      n  correct  errors % correct
0                      positives: identical   3543     3397     146     95.9%
1                  positives: non-identical    843      680     163     80.7%
2     neg: case1 (same street, diff number)  14448    14200     248     98.3%
3  neg: case2 (same number, similar street)     15        6       9     40.0%
4                         neg: case3 (easy)  23942    23916      26     99.9%
5   neg: case2s (swapped look-alike street)   1053      781     272     74.2%
6                        overall (accuracy)  43844    42980     864     98.0%
==================== seed:1 ====================


                                      slice      n  correct  errors % correct
0                      positives: identical   3543     3385     158     95.5%
1                  positives: non-identical    843      657     186     77.9%
2     neg: case1 (same street, diff number)  14448    14224     224     98.4%
3  neg: case2 (same number, similar street)     15        7       8     46.7%
4                         neg: case3 (easy)  23942    23928      14     99.9%
5   neg: case2s (swapped look-alike street)   1053      795     258     75.5%
6                        overall (accuracy)  43844    42996     848     98.1%
==================== seed:2 ====================


                                      slice      n  correct  errors % correct
0                      positives: identical   3543     3542       1    100.0%
1                  positives: non-identical    843      664     179     78.8%
2     neg: case1 (same street, diff number)  14448    13882     566     96.1%
3  neg: case2 (same number, similar street)     15        6       9     40.0%
4                         neg: case3 (easy)  23942    23915      27     99.9%
5   neg: case2s (swapped look-alike street)   1053      778     275     73.9%
6                        overall (accuracy)  43844    42787    1057     97.6%


### 14c. ESIM settings vs siamese (mean pooling) vs XGBoost

`case2s` dataset, every model trained on the `case2s` train split; test, val in brackets; % found for matches, % rejected for non-matches. XGBoost, siamese and the ESIM AUCs copied from `results.md` → *Overture `case2s`*; ESIM slice numbers = min–max over 3 seeds from the 14e output below (siamese and XGBoost = one run each). AUC for `mc1`, `ft_mc1`, `stack_mc1`: Diagnostics D9. Case2 left out (15 test rows).

| Model | Step | Identical matches — % found | **Non-identical matches — % found** | Case1 — % rejected | **Case2s — % rejected** | Easy — % rejected | AUC, non-identical vs case2s |
|---|---|---|---|---|---|---|---|
| XGBoost | 1b | 100.0 (100.0) | **69.9** (76.3) | 99.6 (99.5) | **88.5** (90.2) | 100.0 (100.0) | 0.902 (0.931) |
| Siamese BiLSTM (mean pooling, seed 0) | 3 | 100.0 (100.0) | **80.1** (83.1) | 99.3 (99.3) | **75.7** (74.5) | 100.0 (100.0) | 0.854 (0.871) |
| ESIM, word2vec `min_count=2` (initial) (3 seeds) | 4 | 95.5–100.0 (97.9–100.0) | **77.9–80.7** (80.9–82.7) | 96.1–98.4 (98.2–99.1) | **73.9–75.5** (76.1–80.0) | 99.9 (99.9–100.0) | 0.845–0.850 (0.867–0.883) |
| ESIM, word2vec `min_count=1` (3 seeds) | 4 | 96.8–100.0 (98.8–100.0) | **76.6–78.4** (78.8–81.1) | 95.7–98.3 (98.1–98.7) | **72.1–79.6** (72.7–83.6) | 99.9 (100.0) | 0.814–0.855 (0.840–0.881) |
| ESIM, fastText `min_count=1` (3 seeds) | 4 | 100.0 (100.0) | **76.9–78.4** (79.9–81.6) | 97.7–99.0 (98.2–98.9) | **68.9–72.1** (64.3–71.0) | 99.9–100.0 (100.0) | 0.792–0.834 (0.825–0.855) |
| ESIM, word2vec `min_count=2` + random row per rare token (3 seeds) | 4 | 99.9–100.0 (100.0) | **74.9–78.9** (77.8–81.8) | 98.6–98.9 (99.0–99.2) | **72.9–75.9** (74.6–82.2) | 99.9–100.0 (100.0) | 0.815–0.842 (0.856–0.878) |
| ESIM, stacked `[word2vec mc1; fastText mc1]` (3 seeds) | 4 | 100.0 (100.0) | **75.3–78.4** (78.9–80.8) | 97.6–99.6 (98.6–99.4) | **77.8–80.7** (78.3–84.5) | 99.9–100.0 (99.9–100.0) | 0.851–0.869 (0.875–0.896) |
| *test n* | | *3,543* | *843* | *14,448* | *1,053* | *23,942* | *843 vs 1,053* |


### 14d. Embedding settings side by side, per slice and seed
- `mc1 (min_count =1)`, did not fix the **house_number** isssues with `positives:identical`, since its effect was not 100%, because some same-street numbers have near-identical vectors
- `ft_mc1` fixed the `positives_identical` issue
- `mc1/mc2` > `ft_mc1` for `case2s`


In [ ]:
# 14d: % correct per slice, one column per (setting, seed)  (set up by Claude 2026-11-05; fastText added 2026-11-06)
# slice_table from training.py; the three settings trained before the extension (14e has all five)
SETTINGS_02 = {k: SETTINGS[k] for k in ["mc2", "mc1", "ft_mc1"]}   # also read by D5

for name, split_df, labels in [("val", val, val_label), ("test", test, test_label)]:
    print(f"{'='*20} {name} {'='*20}")
    display(slice_table(split_df, labels, name, SETTINGS_02))


==================== val ====================


mc2                    mc1                  ft_mc1                
                                              0      1       2       0       1       2       0       1       2
slice                                                                                                         
positives: identical                      98.6%  97.9%  100.0%   99.8%   98.8%  100.0%  100.0%  100.0%  100.0%
positives: non-identical                  82.7%  81.8%   80.9%   78.8%   80.3%   81.1%   79.9%   80.2%   81.6%
neg: case1 (same street, diff number)     98.3%  99.1%   98.2%   98.3%   98.7%   98.1%   98.4%   98.9%   98.2%
neg: case2 (same number, similar street)  55.9%  55.9%   55.9%   44.1%   52.9%   50.0%   50.0%   50.0%   50.0%
neg: case3 (easy)                         99.9%  99.9%  100.0%  100.0%  100.0%  100.0%  100.0%  100.0%  100.0%
neg: case2s (swapped look-alike street)   76.1%  79.8%   80.0%   72.7%   83.6%   74.4%   64.3%   71.0%   67.0%
overall (accuracy)                        98.4%  98.7%   98.5%   98.4%   98.7%   98.4%   98.3%   98.6%   98.3%

==================== test ====================


mc2                   mc1                 ft_mc1                
                                              0      1       2      0      1       2       0       1       2
slice                                                                                                       
positives: identical                      95.9%  95.5%  100.0%  99.5%  96.8%  100.0%  100.0%  100.0%  100.0%
positives: non-identical                  80.7%  77.9%   78.8%  76.6%  78.4%   78.4%   77.3%   76.9%   78.4%
neg: case1 (same street, diff number)     98.3%  98.4%   96.1%  97.3%  98.3%   95.7%   98.5%   99.0%   97.7%
neg: case2 (same number, similar street)  40.0%  46.7%   40.0%  26.7%  26.7%   33.3%   33.3%   60.0%   46.7%
neg: case3 (easy)                         99.9%  99.9%   99.9%  99.9%  99.9%   99.9%  100.0%  100.0%   99.9%
neg: case2s (swapped look-alike street)   74.2%  75.5%   73.9%  72.1%  79.6%   74.5%   68.9%   72.1%   70.5%
overall (accuracy)                        98.0%  98.1%   97.6%  97.9%  98.2%   97.5%   98.3%   98.5%   98.1%

### 14e. All five settings, per slice and seed
Running basic checks on results of all the current splits now
- `rand_mc2` was chosen because it was the only one that passed the criteria, which was that it could handle the `UNK` number problem.
- There are some distributions that seem like they could indicate something, but within 1-2%, and a sample size of 3, the standard error is indistinguishable from noise.


In [ ]:
# % correct per slice, one column per (setting, seed)
for name in ["val", "test"]:
    print(f"{'='*20} {name} {'='*20}")
    display(slice_table(*splits[name], name))

==================== val ====================


mc2                    mc1                  ft_mc1                 rand_mc2                 stack_mc1                
                                              0      1       2       0       1       2       0       1       2        0       1       2         0       1       2
slice                                                                                                                                                            
positives: identical                      98.6%  97.9%  100.0%   99.8%   98.8%  100.0%  100.0%  100.0%  100.0%   100.0%  100.0%  100.0%    100.0%  100.0%  100.0%
positives: non-identical                  82.7%  81.8%   80.9%   78.8%   80.3%   81.1%   79.9%   80.2%   81.6%    77.8%   81.8%   80.3%     80.8%   79.3%   78.9%
neg: case1 (same street, diff number)     98.3%  99.1%   98.2%   98.3%   98.7%   98.1%   98.4%   98.9%   98.2%    99.0%   99.0%   99.2%     98.6%   99.4%   99.0%
neg: case2 (same number, similar street)  55.9%  55.9%   55.9%   44.1%   52.9%   50.0%   50.0%   50.0%   50.0%    55.9%   55.9%   55.9%     50.0%   50.0%   58.8%
neg: case3 (easy)                         99.9%  99.9%  100.0%  100.0%  100.0%  100.0%  100.0%  100.0%  100.0%   100.0%  100.0%  100.0%     99.9%  100.0%  100.0%
neg: case2s (swapped look-alike street)   76.1%  79.8%   80.0%   72.7%   83.6%   74.4%   64.3%   71.0%   67.0%    74.6%   76.0%   82.2%     78.3%   79.6%   84.5%
overall (accuracy)                        98.4%  98.7%   98.5%   98.4%   98.7%   98.4%   98.3%   98.6%   98.3%    98.7%   98.8%   98.9%     98.6%   98.9%   98.9%

==================== test ====================


mc2                   mc1                 ft_mc1                 rand_mc2                stack_mc1                
                                              0      1       2      0      1       2       0       1       2        0       1      2         0       1       2
slice                                                                                                                                                         
positives: identical                      95.9%  95.5%  100.0%  99.5%  96.8%  100.0%  100.0%  100.0%  100.0%    99.9%  100.0%  99.9%    100.0%  100.0%  100.0%
positives: non-identical                  80.7%  77.9%   78.8%  76.6%  78.4%   78.4%   77.3%   76.9%   78.4%    74.9%   78.9%  78.2%     78.4%   76.7%   75.3%
neg: case1 (same street, diff number)     98.3%  98.4%   96.1%  97.3%  98.3%   95.7%   98.5%   99.0%   97.7%    98.9%   98.9%  98.6%     97.6%   99.6%   98.0%
neg: case2 (same number, similar street)  40.0%  46.7%   40.0%  26.7%  26.7%   33.3%   33.3%   60.0%   46.7%    33.3%   26.7%  46.7%     46.7%   46.7%   46.7%
neg: case3 (easy)                         99.9%  99.9%   99.9%  99.9%  99.9%   99.9%  100.0%  100.0%   99.9%   100.0%   99.9%  99.9%     99.9%  100.0%   99.9%
neg: case2s (swapped look-alike street)   74.2%  75.5%   73.9%  72.1%  79.6%   74.5%   68.9%   72.1%   70.5%    72.9%   75.9%  74.1%     80.2%   77.8%   80.7%
overall (accuracy)                        98.0%  98.1%   97.6%  97.9%  98.2%   97.5%   98.3%   98.5%   98.1%    98.4%   98.6%  98.4%     98.3%   98.8%   98.3%

### 14f. Rare numbers, all five settings
- When isolating for `UNK` numbers, `rand_mc2` is the only one that passed the threshold of `145/161` matches for `case1`

In [ ]:
# rare-number group (02's D2-D5): identical positives found / case1 negatives rejected inside it, errors outside it
rare_number_table({"val": splits["val"], "test": splits["test"]})

identical UNK: found case1 UNK: rejected  identical other: missed  case1 other: errors
split setting   seed                                                                                       
val   mc2       0                 29 / 78           141 / 161                        0                  216
                1                  9 / 78           158 / 161                        1                  124
                2                 78 / 78             0 / 161                        0                   92
      mc1       0                 72 / 78            21 / 161                        0                   98
                1                 38 / 78           111 / 161                        1                  125
                2                 78 / 78            12 / 161                        0                  116
      ft_mc1    0                 78 / 78           151 / 161                        0                  213
                1                 78 / 78           151 / 161                        0                  137
                2                 78 / 78           144 / 161                        0                  229
      rand_mc2  0                 78 / 78           158 / 161                        0                  128
                1                 78 / 78           155 / 161                        0                  132
                2                 78 / 78           154 / 161                        0                  100
      stack_mc1 0                 78 / 78           126 / 161                        0                  155
                1                 78 / 78           148 / 161                        0                   72
                2                 78 / 78           128 / 161                        0                  102
test  mc2       0                32 / 178           381 / 404                        0                  225
                1                20 / 178           387 / 404                        0                  207
                2               178 / 178             0 / 404                        1                  162
      mc1       0               162 / 178           149 / 404                        1                  129
                1                72 / 178           316 / 404                        7                  160
                2               178 / 178            64 / 404                        0                  276
      ft_mc1    0               178 / 178           376 / 404                        0                  194
                1               178 / 178           388 / 404                        0                  124
                2               178 / 178           350 / 404                        0                  278
      rand_mc2  0               178 / 178           385 / 404                        3                  143
                1               178 / 178           392 / 404                        0                  152
                2               178 / 178           379 / 404                        2                  171
      stack_mc1 0               178 / 178           291 / 404                        0                  232
                1               178 / 178           386 / 404                        0                   40
                2               178 / 178           310 / 404                        0                  202

### 14g. Decision
**Result (David, 2026-11-08; written down by Claude from the discussion):**

**Decision rule, fixed before reading val:**

| slice | role | why |
|---|---|---|
| identical positives | ignored | an exact-string pre-check handles them before the model |
| case1, rare-number group | must-pass: ≥ 145 / 161 rejected (≥ 90%) on every val seed | reading house numbers is the minimum; the pre-check does not remove these pairs, they differ as strings |
| non-identical positives | decides | |
| `case2s` | decides (hardest slice) | |
| case2 (real, n = 34 val) | ignored | too small, seed noise |
| case3 (easy) | sanity check only | |

A gap counts only if it is bigger than the seed spread (min–max over 3 seeds). Note: non-identical and `case2s` pull in opposite directions (a stricter model gains `case2s` and loses rewrites), so a model that only leans toward "no" wins one and loses the other.

**Must-pass (val, `case1 UNK: rejected`):** `mc2` 141 / 158 / 0, `mc1` 21 / 111 / 12, `ft_mc1` 151 / 151 / 144, `rand_mc2` 158 / 155 / 154, `stack_mc1` 126 / 148 / 128. **Only `rand_mc2` passes** (`ft_mc1` misses by one pair on seed 2; the cut was not moved).

**Why random rows work:** the group's question is identity ("same number?"), not meaning. A fixed random row per dropped token gives the same vector for the same token and a near-orthogonal one for a different token, so alignment finds an exact partner or none. word2vec places same-street numbers close (cos 0.92), fastText fairly close (0.79), and stacking the two still says "close".

**Cost vs `mc2` on the deciding slices (val, seeds 0 / 1 / 2):** non-identical 82.7 / 81.8 / 80.9 vs 77.8 / 81.8 / 80.3; `case2s` 76.1 / 79.8 / 80.0 vs 74.6 / 76.0 / 82.2. Ranges overlap on both. **Conclusion: `rand_mc2` fixes the rare-number group; no difference on non-identical or `case2s` is detectable at 3 seeds, a cost of about 1 point is not ruled out** (more seeds would settle it; val's ~800–900 rows per slice also limit resolution to about ±1.4 pts).

**`results.md`:** `mc2` (the reproduction) and `rand_mc2` (the fix) as test min–max (val min–max); the other three settings get one sentence each in the Step 4 paragraph.


## Diagnostics
One-off inspections, kept out of the pipeline cells above. Results are written into the markdown (question → result → query).

### D1. Why do seeds 0 and 1 reject ~150 identical positives?
- We want to figure out why we're dropping rows for `positive:identical` when they're literally the same.
- Results showed it's because the `House_Number` was `<unk>`, which made it incredibly difficult for it to differentiate cases in where `House Number` is different vs same

In [ ]:
# Group = numbers that were UNK in the original run, fixed across settings so mc1 vs mc2 compare the same rows
group_vocab = Word2Vec.load("checkpoints/word2vec_50_mc2.model").wv.key_to_index
for p in [0, 1]:
    preds = res_test[0]   # pick the seed here; the mask and the table must use the same predictions
    mask = (test["address_a"] == test["address_b"]).values & (test_label == 1).values & (preds == p)

    df = find_random_sample_given_mask(test, preds, test_label, mask)

    for check in ["a", "b"]:
        df[f"address_{check}_ids"] = df[f"address_{check}"].apply(lambda x: [group_vocab.get(word) for word in tokenize(x)])

    display(df[["address_a", "address_b", "predicted", "label", "address_a_ids", "address_b_ids"]])

,address_a,address_b,predicted,label,address_a_ids,address_b_ids
748,2436 MAIN ST,2436 MAIN ST,0,1,"[None, 3, 0]","[None, 3, 0]"
813,4031 MAIN ST,4031 MAIN ST,0,1,"[None, 3, 0]","[None, 3, 0]"
817,4008 MAIN ST,4008 MAIN ST,0,1,"[None, 3, 0]","[None, 3, 0]"
3756,3123 BERLIN TPKE,3123 BERLIN TPKE,0,1,"[None, 46, 7]","[None, 46, 7]"
3858,3440 BERLIN TPKE,3440 BERLIN TPKE,0,1,"[None, 46, 7]","[None, 46, 7]"
852,2979 MADISON AVE,2979 MADISON AVE,0,1,"[None, 430, 2]","[None, 430, 2]"
840,4675 MAIN ST,4675 MAIN ST,0,1,"[None, 3, 0]","[None, 3, 0]"
809,4175 MAIN ST,4175 MAIN ST,0,1,"[None, 3, 0]","[None, 3, 0]"
3734,2691 BERLIN TPKE,2691 BERLIN TPKE,0,1,"[None, 46, 7]","[None, 46, 7]"
816,4060 MAIN ST,4060 MAIN ST,0,1,"[None, 3, 0]","[None, 3, 0]"


,address_a,address_b,predicted,label,address_a_ids,address_b_ids
1858,2249 NEW LONDON TPKE,2249 NEW LONDON TPKE,1,1,"[1902, 15, 66, 7]","[1902, 15, 66, 7]"
1606,201 SAYBROOK RD,201 SAYBROOK RD,1,1,"[312, 339, 1]","[312, 339, 1]"
1506,9 BRITTON LN,9 BRITTON LN,1,1,"[82, 2974, 26]","[82, 2974, 26]"
892,201 HIGHLAND AVE,201 HIGHLAND AVE,1,1,"[312, 176, 2]","[312, 176, 2]"
2474,18 KINNEY HOLLOW RD,18 KINNEY HOLLOW RD,1,1,"[135, 2391, 342, 1]","[135, 2391, 342, 1]"
900,89 ROWAYTON AVE,89 ROWAYTON AVE,1,1,"[250, 920, 2]","[250, 920, 2]"
613,3 OLD TOWN PARK RD,3 OLD TOWN PARK RD,1,1,"[62, 23, 118, 18, 1]","[62, 23, 118, 18, 1]"
2302,927 BLUE HILLS AVE,927 BLUE HILLS AVE,1,1,"[1841, 479, 133, 2]","[1841, 479, 133, 2]"
4196,209 MAPLE AVE,209 MAPLE AVE,1,1,"[448, 79, 2]","[448, 79, 2]"
3764,1616 WILLARD AVE,1616 WILLARD AVE,1,1,"[2701, 519, 2]","[2701, 519, 2]"


### D2. Is the `UNK` number the pattern across the whole slice? 
- Expansion on D1, to see if this is a systemic issue
- Results show that for all seeds, and example, when it's **comparing None Numbers** is when there's a result split, and when that's not the case, there is NO RESULT SPLIT


In [ ]:
# D2: UNK-number share, missed vs found identical positives
extract_non_number = lambda x: x.str.replace(r"^(\d+\s+)", "", regex= True)   # defined before the loop uses it
for seed in [0, 1, 2]:
    preds = res_val[seed]   # pick the seed here; the mask and the table must use the same predictions

    
    # mask = (val["address_a"] == val["address_b"]).values & (val_label == 1).values & (preds == p)
    mask = (val["address_a"] == val["address_b"]).values

    df = find_random_sample_given_mask(val, preds, val_label, mask, size = None)

    
    # Add Tokenized List 
    for check in ["a", "b"]:
        #Embedded Tokens
        df[f"address_{check}_ids"] = df[f"address_{check}"].apply(lambda x: [group_vocab.get(word) for word in tokenize(x)])
        #Starts With Number?
        df[f"address_{check}_num_start"] = df[f"address_{check}"].str.contains(r"^\d+")
        df[f"non_number_{check}"] = extract_non_number(df[f"address_{check}"])

    
    df["compare_none_numbers"] = (
        (df["address_a_ids"].str[0].isna()) &
        (df["address_b_ids"].str[0].isna()) & 
        (df["address_a_num_start"] == True) &
        (df["address_b_num_start"] == True) & 
        (df["non_number_a"] == df["non_number_b"])
    )
    display_list = [
        "address_a", "address_b", 
        "predicted", "label", 
        "address_a_ids", "address_b_ids", 
        "address_a_num_start", "address_b_num_start", 
        "non_number_a","non_number_b", 
        "compare_none_numbers"]

    # display(df[display_list])
    print(f"{'='*20} seed:{seed} {'='*20}")
    display(df.groupby(["compare_none_numbers", "predicted"])["predicted"].size().unstack("predicted").sort_index(ascending = False))

==================== seed:0 ====================

predicted,0,1
compare_none_numbers,,
True,49.0,29.0
False,NaN,3312.0


==================== seed:1 ====================


predicted,0,1
compare_none_numbers,,
True,69,9
False,1,3311


==================== seed:2 ====================


predicted,1
compare_none_numbers,
True,78
False,3312


### D3. Does seed 2 pay for its 100% on case1? (val, case1 negatives)
**Notes (David):**
Since we've concluded that the model struggles to compare addresses when house_number except `<unk>` (other than seed 2), we would expect this to carry over for negatives as well
- As shown, `seed0` and `seed1` seem to be able differentiate quite well.
- In comparison `seed2` categorizes all negative values as 

In [ ]:
for seed in [0, 1, 2]:
    preds = res_val[seed]
    mask = (val["neg_type"] == "case1").values & (val_label == 0).values   # D2 had: identical strings

    df = find_random_sample_given_mask(val, preds, val_label, mask, size = None)

    for check in ["a", "b"]:
        df[f"address_{check}_ids"] = df[f"address_{check}"].apply(lambda x: [group_vocab.get(word) for word in tokenize(x)])
        df[f"address_{check}_num_start"] = df[f"address_{check}"].str.contains(r"^\d+")
        df[f"non_number_{check}"] = extract_non_number(df[f"address_{check}"])

    df["compare_none_numbers"] = (
        (df["address_a_ids"].str[0].isna()) &
        (df["address_b_ids"].str[0].isna()) &
        (df["address_a_num_start"] == True) &
        (df["address_b_num_start"] == True) &
        (df["non_number_a"] == df["non_number_b"])
    )

    print(f"{'='*20} seed:{seed} {'='*20}")
    display(df.groupby(["compare_none_numbers", "predicted"])["predicted"].size().unstack("predicted").sort_index(ascending = False))


==================== seed:0 ====================


predicted,0,1
compare_none_numbers,,
True,141,20
False,13404,216


==================== seed:1 ====================


predicted,0,1
compare_none_numbers,,
True,158,3
False,13496,124


==================== seed:2 ====================


predicted,0,1
compare_none_numbers,,
True,NaN,161.0
False,13528.0,92.0


### D4, confirmation on test set
- Same Test as D2 + D3, but on the test_dataset
- Confirms the same

In [ ]:
# D4: D2 (identical positives) and D3 (case1 negatives) on test (written by Claude 2026-11-05)
# Needs extract_non_number from D2 and res_test from 14b.
def unk_number_table(df_split, labels, preds, mask):
    df = find_random_sample_given_mask(df_split, preds, labels, mask, size = None)
    for check in ["a", "b"]:
        df[f"address_{check}_ids"] = df[f"address_{check}"].apply(lambda x: [group_vocab.get(word) for word in tokenize(x)])
        df[f"address_{check}_num_start"] = df[f"address_{check}"].str.contains(r"^\d+")
        df[f"non_number_{check}"] = extract_non_number(df[f"address_{check}"])
    df["compare_none_numbers"] = (
        (df["address_a_ids"].str[0].isna()) &
        (df["address_b_ids"].str[0].isna()) &
        (df["address_a_num_start"] == True) &
        (df["address_b_num_start"] == True) &
        (df["non_number_a"] == df["non_number_b"])
    )
    return df.groupby(["compare_none_numbers", "predicted"])["predicted"].size().unstack("predicted").sort_index(ascending = False)

row_groups = {
    "D2 identical positives (1 = correct)": (test["address_a"] == test["address_b"]).values & (test_label == 1).values,
    "D3 case1 negatives (0 = correct)":     (test["neg_type"] == "case1").values & (test_label == 0).values,
}
for name, mask in row_groups.items():
    for seed in [0, 1, 2]:
        print(f"{'='*10} {name} | seed:{seed} {'='*10}")
        display(unk_number_table(test, test_label, res_test[seed], mask))


========== D2 identical positives (1 = correct) | seed:0 ==========


predicted,0,1
compare_none_numbers,,
True,146.0,32.0
False,NaN,3365.0


========== D2 identical positives (1 = correct) | seed:1 ==========


predicted,0,1
compare_none_numbers,,
True,158.0,20.0
False,NaN,3365.0


========== D2 identical positives (1 = correct) | seed:2 ==========


predicted,0,1
compare_none_numbers,,
True,NaN,178.0
False,1.0,3364.0


... (2 outputs left out)

========== D3 case1 negatives (0 = correct) | seed:1 ==========


predicted,0,1
compare_none_numbers,,
True,387,17
False,13837,207


========== D3 case1 negatives (0 = correct) | seed:2 ==========


predicted,0,1
compare_none_numbers,,
True,NaN,404.0
False,13882.0,162.0


### D5. Health Check + Findings Summary
- Health check: `broken_run_problems` on every saved history
    - Checks if any part of history is `NA`
    - Make sure `first_val_loss` <= 0.693 (otherwise would indicate lack of learning)
        - `-log(0.5) = 0.693` ~ BCE bullshit
    - min_train_loss < 0.30
        - Since a pure guess of False (9:1), gets us a loss of about `0.325`, we want to ensure our BEST result is better than the best guess
        - `always guessing negative on a 1:9 dataset = 0.325 (BCE)`
    - Tests to check directly if it's just guessing no.
- Same `UNK` group as D2–D4 (both numbers unknown to the `min_count=2` vocab, rest of the address identical), counted per setting and seed: identical positives found, case1 negatives rejected, and errors outside the group
- Test is confirmation only; the choice is made on val

In [ ]:
# D5: health check + D2/D3 group across settings (written by Claude 2026-11-06)
# Needs SETTINGS_02 (14d), group_vocab (D1), extract_non_number (D2).
for name, folder in SETTINGS_02.items():
    for seed in [0, 1, 2]:
        h = pd.read_csv(f"{folder}/history_seed_{seed}.csv")
        print(f"{name:7s} seed {seed}: {len(h)} epochs, problems: {broken_run_problems(h)}")

def unk_group(split_df):
    """Both addresses start with a number unknown to the min_count=2 vocab, rest of the address identical."""
    grp = np.ones(len(split_df), dtype=bool)
    for side in ["a", "b"]:
        addr = split_df[f"address_{side}"]
        first_unknown = addr.apply(lambda x: group_vocab.get((tokenize(x) or [None])[0]) is None)
        grp &= first_unknown.values & addr.str.contains(r"^\d+").values
    return grp & (extract_non_number(split_df["address_a"]) == extract_non_number(split_df["address_b"])).values

rows = []
for split, split_df, labels in [("val", val, val_label), ("test", test, test_label)]:
    grp = unk_group(split_df)
    ident = (split_df["address_a"] == split_df["address_b"]).values & (labels == 1).values
    case1 = (split_df["neg_type"] == "case1").values & (labels == 0).values
    for name, folder in SETTINGS_02.items():
        for seed in [0, 1, 2]:
            p = np.load(f"{folder}/preds_{split}_seed_{seed}.npy")
            rows.append({
                "split": split, "setting": name, "seed": seed,
                "identical UNK: found":   f"{(p[ident & grp] == 1).sum()} / {(ident & grp).sum()}",
                "case1 UNK: rejected":    f"{(p[case1 & grp] == 0).sum()} / {(case1 & grp).sum()}",
                "identical other: missed": (p[ident & ~grp] == 0).sum(),
                "case1 other: errors":     (p[case1 & ~grp] == 1).sum(),
            })
display(pd.DataFrame(rows).set_index(["split", "setting", "seed"]))


mc2     seed 0: 11 epochs, problems: []
mc2     seed 1: 9 epochs, problems: []
mc2     seed 2: 8 epochs, problems: []
mc1     seed 0: 5 epochs, problems: []
mc1     seed 1: 12 epochs, problems: []
mc1     seed 2: 8 epochs, problems: []
ft_mc1  seed 0: 5 epochs, problems: []
ft_mc1  seed 1: 8 epochs, problems: []
ft_mc1  seed 2: 8 epochs, problems: []


identical UNK: found case1 UNK: rejected  identical other: missed  case1 other: errors
split setting seed                                                                                       
val   mc2     0                 29 / 78           141 / 161                        0                  216
              1                  9 / 78           158 / 161                        1                  124
              2                 78 / 78             0 / 161                        0                   92
      mc1     0                 72 / 78            21 / 161                        0                   98
              1                 38 / 78           111 / 161                        1                  125
              2                 78 / 78            12 / 161                        0                  116
      ft_mc1  0                 78 / 78           151 / 161                        0                  213
              1                 78 / 78           151 / 161                        0                  137
              2                 78 / 78           144 / 161                        0                  229
test  mc2     0                32 / 178           381 / 404                        0                  225
              1                20 / 178           387 / 404                        0                  207
              2               178 / 178             0 / 404                        1                  162
      mc1     0               162 / 178           149 / 404                        1                  129
              1                72 / 178           316 / 404                        7                  160
              2               178 / 178            64 / 404                        0                  276
      ft_mc1  0               178 / 178           376 / 404                        0                  194
              1               178 / 178           388 / 404                        0                  124
              2               178 / 178           350 / 404                        0                  278

### D6. Swap test (`rand_mc2`, val). 
- Check if we have any preexisting reliance on ordering, regarding address side
- Results show flipped predictions of `2.3–4.3%` for `non-identical` and `2.3–4.1%` for `case2s`. case1 `0.1–0.5%`, easy ≈ 0, overall `0.15–0.34%`. Largest probability change 0.36–0.56: the flips are pairs near the threshold. Not a huge amount, but indicates some learned behavior

In [ ]:
swapped = val.rename(columns={"address_a": "address_b", "address_b": "address_a"})
loader_ab = make_loaders(vocab_rand, {"val": (val, val_label)})["val"]
loader_ba = make_loaders(vocab_rand, {"val": (swapped, val_label)})["val"]

rows = []
for seed in SEEDS:
    ckpt = torch.load(f"{SETTINGS['rand_mc2']}/esim_seed_{seed}.pt")
    model = ESIM(matrix_rand)
    model.load_state_dict(ckpt["state_dict"])
    p_ab = predict(model, loader_ab)            # probabilities, (a, b)
    p_ba = predict(model, loader_ba)            # probabilities, (b, a)
    ab = (p_ab >= THRESHOLD).int().numpy()
    ba = (p_ba >= THRESHOLD).int().numpy()
    assert np.array_equal(ab, np.load(f"{SETTINGS['rand_mc2']}/preds_val_seed_{seed}.npy")), "unswapped run != saved predictions"
    flips = ab != ba
    rows.append({"seed": seed, "flips": int(flips.sum()), "of": len(flips), "% flipped": 100 * flips.mean(),
                 "max |p_ab - p_ba|": (p_ab - p_ba).abs().max().item(),
                 **{f"flips {t}": int(flips[(val["neg_type"] == t).values].sum()) for t in sorted(val["neg_type"].unique())}})
pd.DataFrame(rows).set_index("seed")


,flips,of,% flipped,max |p_ab - p_ba|,flips case0,flips case1,flips case2,flips case2s,flips case3
seed,,,,,,,,,
0,68,41865,0.162427,0.543069,25,14,0,26,3
1,144,41865,0.343963,0.564962,34,67,0,37,6
2,63,41865,0.150484,0.358619,18,21,0,21,3


### D7. Symmetric prediction (`rand_mc2`, val). 
- We just check if running a version which averages both symmetric representations, provides a meaningful improvement to scores.
- Results show that there are very few results


In [ ]:
from helpers.check_model_performance import review_typical_slices

class SymmetricESIM(nn.Module):
    """Wraps a trained ESIM; logit = mean of both input orders."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, ids_a, len_a, ids_b, len_b):
        return (self.model(ids_a, len_a, ids_b, len_b) + self.model(ids_b, len_b, ids_a, len_a)) / 2

# test on one batch: symmetric output is identical for swapped inputs
ids_a, len_a, ids_b, len_b, _ = next(iter(loader_ab))
ckpt = torch.load(f"{SETTINGS['rand_mc2']}/esim_seed_0.pt")
m = ESIM(matrix_rand); m.load_state_dict(ckpt["state_dict"]); m.eval()
sym = SymmetricESIM(m)
with torch.no_grad():
    assert torch.allclose(sym(ids_a, len_a, ids_b, len_b), sym(ids_b, len_b, ids_a, len_a))
print("symmetric on swapped inputs: OK")

cols, changed = {}, {}
for seed in SEEDS:
    ckpt = torch.load(f"{SETTINGS['rand_mc2']}/esim_seed_{seed}.pt")
    model = ESIM(matrix_rand)
    model.load_state_dict(ckpt["state_dict"])
    plain = np.load(f"{SETTINGS['rand_mc2']}/preds_val_seed_{seed}.npy")
    symm = (predict(SymmetricESIM(model), loader_ab) >= THRESHOLD).int().numpy()
    changed[seed] = int((plain != symm).sum())
    for name, preds in [("plain", plain), ("symmetric", symm)]:
        cols[(seed, name)] = review_typical_slices(val, preds, val_label).set_index("slice")["% correct"]
print("predictions changed by averaging, per seed:", changed)
pd.DataFrame(cols)


symmetric on swapped inputs: OK
predictions changed by averaging, per seed: {0: 36, 1: 63, 2: 26}


0                 1                 2          
                                           plain symmetric   plain symmetric   plain symmetric
slice                                                                                         
positives: identical                      100.0%    100.0%  100.0%    100.0%  100.0%    100.0%
positives: non-identical                   77.8%     77.9%   81.8%     82.2%   80.3%     79.7%
neg: case1 (same street, diff number)      99.0%     99.0%   99.0%     98.9%   99.2%     99.3%
neg: case2 (same number, similar street)   55.9%     55.9%   55.9%     55.9%   55.9%     55.9%
neg: case3 (easy)                         100.0%    100.0%  100.0%    100.0%  100.0%    100.0%
neg: case2s (swapped look-alike street)    74.6%     74.6%   76.0%     76.3%   82.2%     82.0%
overall (accuracy)                         98.7%     98.7%   98.8%     98.7%   98.9%     98.9%

### D8. AUC, non-identical vs `case2s`, setting `mc2`
(Claude, 2026-11-11, for `results.md`'s AUC column.) Probabilities saved as `probs_{split}_seed_{s}.npy` in `esim_mc2/` (`>= 0.5` reproduces the saved predictions). Test (val): 0.849 (0.867) / 0.845 (0.882) / 0.850 (0.883). `rand_mc2`'s AUC is in `transformers/01_bi_encoder.ipynb` D3–D4. Query below.


In [ ]:
from sklearn.metrics import roc_auc_score
vocab_mc2_t, matrix_mc2_t = plain_table("word2vec", 2)
loaders_mc2 = make_loaders(vocab_mc2_t, {"val": (val, val_label), "test": (test, test_label)})
for seed in SEEDS:
    m = ESIM(matrix_mc2_t)
    m.load_state_dict(torch.load(f"{SETTINGS['mc2']}/esim_seed_{seed}.pt", weights_only=False)["state_dict"])
    for split, (df, labels) in {"val": (val, val_label), "test": (test, test_label)}.items():
        path = f"{SETTINGS['mc2']}/probs_{split}_seed_{seed}.npy"
        if not os.path.exists(path):
            np.save(path, predict(m, loaders_mc2[split]).numpy())
        p = np.load(path)
        pos = (labels.values == 1) & (df["address_a"] != df["address_b"]).values
        c2s = (df["neg_type"] == "case2s").values
        print(seed, split, round(roc_auc_score(np.r_[np.ones(pos.sum()), np.zeros(c2s.sum())], np.r_[p[pos], p[c2s]]), 3))


NameError: name 'plain_table' is not defined

### D9. AUC, non-identical vs `case2s`, all five settings
(Claude, 2026-11-18, for 14c's AUC column.) D8's method for the three settings that had no saved probabilities (`mc1`, `ft_mc1`, `stack_mc1`): rebuild each run from its checkpoint, check its vocab is the run's and that `>= 0.5` reproduces the saved predictions, save `probs_{split}_seed_{s}.npy` next to them. Then the AUC for all five settings, per split and seed.

In [ ]:
# D9: probabilities for mc1 / ft_mc1 / stack_mc1 (skipped once saved), then AUC non-identical vs case2s for all five settings
from sklearn.metrics import roc_auc_score
table_builders = {
    "mc1": lambda: plain_table("word2vec", 1),
    "ft_mc1": lambda: plain_table("fasttext", 1),
    "stack_mc1": lambda: stacked_table(min_count=1),
}
eval_splits = {"val": (val, val_label), "test": (test, test_label)}
for name, build in table_builders.items():
    if all(os.path.exists(f"{SETTINGS[name]}/probs_{split}_seed_{seed}.npy") for split in eval_splits for seed in SEEDS):
        continue
    vocab_s, matrix_s = build()
    loaders_s = make_loaders(vocab_s, eval_splits)
    for seed in SEEDS:
        ckpt_s = torch.load(f"{SETTINGS[name]}/esim_seed_{seed}.pt", weights_only=False)
        assert ckpt_s["vocab"] == vocab_s, (name, seed, "vocab differs from the one the run trained with")
        m = ESIM(matrix_s)
        m.load_state_dict(ckpt_s["state_dict"])
        for split in eval_splits:
            p = predict(m, loaders_s[split]).numpy()
            saved = np.load(f"{SETTINGS[name]}/preds_{split}_seed_{seed}.npy").ravel()
            assert np.array_equal((p >= THRESHOLD).astype(int), saved), (name, seed, split, "doesn't reproduce the saved predictions")
            np.save(f"{SETTINGS[name]}/probs_{split}_seed_{seed}.npy", p)

def auc_nonident_vs_case2s(df, labels, p):
    pos = (labels.values == 1) & (df["address_a"] != df["address_b"]).values
    c2s = (df["neg_type"] == "case2s").values
    return roc_auc_score(np.r_[np.ones(pos.sum()), np.zeros(c2s.sum())], np.r_[p[pos], p[c2s]])

pd.DataFrame({
    name: {f"{split} seed {seed}": round(auc_nonident_vs_case2s(df, lab, np.load(f"{folder}/probs_{split}_seed_{seed}.npy")), 3)
           for split, (df, lab) in eval_splits.items() for seed in SEEDS}
    for name, folder in SETTINGS.items()
}).T

NameError: name 'val' is not defined

### D10. AUC, case0 vs case1 where one side has no house number (test)
(Claude, 2026-11-18.) Same comparison as `siamese_model/03_siamese_analysis.ipynb`'s "one side missing" AUC: test pairs where exactly one address has no number (`add_number_street`), matches (case0) vs same-street non-matches (case1). Threshold-free, so it shows whether a model *ranks* these better or only sits at a different point of the trade-off. For reference, from that notebook: XGBoost 0.537, siamese mean 0.526, siamese max 0.545.

In [ ]:
# D10: AUC case0 vs case1 within "one side missing" (test), all five settings, per seed
from helpers.pairs import add_number_street
test_num = add_number_street(test)
one_missing = (test_num["number_a"].notna() != test_num["number_b"].notna()).values
mask = one_missing & test_num["neg_type"].isin(["case0", "case1"]).values
lab = test_label.values.ravel()
print(f"n: {int(lab[mask].sum())} case0 / {int((lab[mask] == 0).sum())} case1")
pd.DataFrame({
    name: {f"seed {seed}": round(roc_auc_score(lab[mask], np.load(f"{folder}/probs_test_seed_{seed}.npy")[mask]), 3) for seed in SEEDS}
    for name, folder in SETTINGS.items()
}).T

ModuleNotFoundError: No module named 'helpers'

### D11. AUC, matches vs non-matches inside the rare-number group, all five settings
(Claude, 2026-11-19, at David's request.) Question: do the `min_count` 1 seeds disagree because their scores don't separate the group's matches from its non-matches (AUC near 0.5, so each seed's 0.5 threshold decides), or because they do separate them but at different thresholds (AUC well above 0.5)?

Group = `rare_number_group` (same street, both house numbers outside the word2vec `min_count` 2 vocab), identical matches vs case1 non-matches, as in 14f. For `mc2`, a match and a non-match *on the same street* are the same input (below), so they get the same score. Across streets the scores still differ (the street words differ), so `mc2`'s AUC need not be exactly 0.5 (correction, Claude 2026-11-19: an earlier version of this cell said it must be):

| Pair | `mc2` model input | Label |
|---|---|---|
| `2147 SUMMER ST` / `2147 SUMMER ST` | `UNK SUMMER ST` / `UNK SUMMER ST` | match |
| `2147 SUMMER ST` / `1867 SUMMER ST` | `UNK SUMMER ST` / `UNK SUMMER ST` | non-match |

Result: *(David)*

In [31]:
# D11: AUC identical matches vs case1 non-matches inside the rare-number group, per setting, split and seed
from sklearn.metrics import roc_auc_score
rows = {}
for split, (df, labels) in {"val": (val, val_label), "test": (test, test_label)}.items():
    grp = rare_number_group(df)
    lab = labels.values.ravel()
    ident = (df["address_a"] == df["address_b"]).values & (lab == 1)
    case1 = (df["neg_type"] == "case1").values & (lab == 0)
    mask = grp & (ident | case1)
    print(f"{split} n: {int((mask & ident).sum())} matches / {int((mask & case1).sum())} non-matches")
    for name, folder in SETTINGS.items():
        rows[(split, name)] = {f"seed {seed}": round(roc_auc_score(lab[mask], np.load(f"{folder}/probs_{split}_seed_{seed}.npy").ravel()[mask]), 3)
                               for seed in SEEDS}
pd.DataFrame(rows).T

val n: 78 matches / 161 non-matches
test n: 178 matches / 404 non-matches


seed 0  seed 1  seed 2
val  mc2         0.679   0.563   0.712
     mc1         0.691   0.776   0.745
     ft_mc1      1.000   1.000   1.000
     rand_mc2    1.000   1.000   1.000
     stack_mc1   1.000   1.000   0.998
test mc2         0.589   0.544   0.537
     mc1         0.761   0.765   0.706
     ft_mc1      0.999   1.000   1.000
     rand_mc2    1.000   1.000   1.000
     stack_mc1   0.999   0.999   1.000